# Démo 5 : un premier classifieur d'images

Objectif : appliquer la boucle de la démo 1 à de vraies images : charger un jeu de données, le découper en entraînement, validation et test, le servir par minibatchs, entraîner un réseau qui reconnaît des vêtements, et suivre l'apprentissage dans TensorBoard.

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- charger un vrai jeu d'images et le découper en entraînement, validation et test ;
- servir les images par minibatchs avec un DataLoader ;
- entraîner un réseau qui reconnaît des vêtements, avec la même boucle qu'au premier neurone, et suivre l'apprentissage dans TensorBoard.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Télécharger Fashion-MNIST

Le téléchargement n'a lieu qu'une fois. `%%capture` masque sa barre de progression.

In [ ]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)
datasets.FashionMNIST("data", train=False, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 2. Charger les images

**torchvision fournit Fashion-MNIST sous forme de Dataset, prêt à l'emploi.**

Un **Dataset** est un objet qui renvoie un exemple (une image et sa classe) quand on lui donne un
numéro. C'est la forme standard des données en PyTorch.

Le code :
- `datasets.FashionMNIST("data", train=True, ...)` : les 60 000 images d'entraînement ; `train=False` donne
  les 10 000 images de test ;
- `transform=transforms.ToTensor()` : chaque image est convertie en tenseur, aux valeurs entre 0 et 1 ;
- `len(...)` : le nombre d'exemples.

In [ ]:
import torch
from torchvision import datasets, transforms

train_complet = datasets.FashionMNIST("data", train=True, transform=transforms.ToTensor())
test_ds = datasets.FashionMNIST("data", train=False, transform=transforms.ToTensor())
print(len(train_complet), len(test_ds))

60000 images d'entraînement et 10000 images de test.

## 3. Une image, un tenseur

**Un exemple du Dataset : une image et le numéro de sa classe.**

Le code :
- `train_complet[0]` : le premier exemple, un couple (image, label) ;
- `image.shape` : la forme de l'image ;
- `train_complet.classes[label]` : le nom de la classe correspondant au numéro.

Prédisez : quelle forme pour une image en niveaux de gris de 28 x 28 pixels ?

In [ ]:
image, label = train_complet[0]
print(image.shape)
print(label, train_complet.classes[label])

Une image est un tenseur `[1, 28, 28]` : un canal de gris, 28 x 28 pixels. Son label est un simple numéro,
9, qui correspond à la classe `Ankle boot`, une bottine.

**À retenir** : le modèle ne voit jamais le nom des classes, seulement leur numéro.

## 4. Découvrir les images (fourni)

**Une image de chaque classe, pour voir ce que le modèle devra reconnaître.**

Le code (à survoler) :
- `plt.subplots(1, 10)` : une rangée de dix cases ;
- pour chaque classe, on cherche la première image de cette classe et on l'affiche avec son nom.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for classe, ax in enumerate(axes):
    index = (train_complet.targets == classe).nonzero()[0].item()
    ax.imshow(train_complet[index][0][0], cmap="gray")
    ax.set_title(train_complet.classes[classe], fontsize=8)
    ax.axis("off")
plt.show()

Dix classes de vêtements et de chaussures, en petites images grises : certaines se ressemblent, comme les
chemises, les t-shirts et les pulls.

## 5. Compter les exemples par classe

**Les classes sont-elles équilibrées ?**

Si une classe domine, un modèle qui la prédit toujours obtient déjà un bon score. Il faut le savoir
avant de juger une accuracy.

Le code :
- `train_complet.targets` : les labels de toutes les images ;
- `torch.bincount(...)` : combien de fois apparaît chaque numéro de classe.

In [ ]:
print(torch.bincount(train_complet.targets))

Exactement 6000 images par classe : le jeu est parfaitement équilibré.

**À retenir** : ici, répondre au hasard donne une chance sur dix ; l'accuracy se lira par rapport à ce repère.

## 6. Entraînement, validation, test

**On sépare les données en trois jeux, chacun avec un rôle.**

Un modèle peut apprendre ses exemples par coeur sans savoir généraliser. On l'entraîne sur
l'**entraînement**, on compare les réglages sur la **validation**, et on garde le **test** pour une seule
mesure finale, sur des images jamais vues.

Le code :
- `random_split(train_complet, [50000, 10000])` : coupe au hasard les 60 000 images en deux jeux ;
- `torch.manual_seed(0)` : le même découpage à chaque exécution.

In [ ]:
from torch.utils.data import random_split

torch.manual_seed(0)
train_ds, val_ds = random_split(train_complet, [50000, 10000])
print(len(train_ds), len(val_ds), len(test_ds))

50000 images pour entraîner, 10000 pour valider, et les 10000 de test mises de côté.

## 7. Les DataLoader : des minibatchs

**Un DataLoader sert les images par paquets, les minibatchs.**

On ne corrige pas le modèle après chaque image, ni après les 50 000 d'un coup : on le corrige après
chaque paquet de 64. C'est plus rapide et plus stable.

Le code :
- `DataLoader(train_ds, batch_size=64, shuffle=True)` : des paquets de 64, dans un ordre mélangé à chaque
  époque ;
- `batch_size=256` pour la validation et le test : on ne fait que mesurer, des paquets plus gros vont plus vite ;
- `next(iter(train_loader))` : le premier paquet, pour regarder sa forme.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

Un minibatch : `[64, 1, 28, 28]`, soit 64 images d'un canal de 28 x 28 pixels, et leurs 64 labels.

**À retenir** : la forme d'un lot d'images est (lot, canaux, hauteur, largeur).

## 8. Le modèle, écrit comme une classe

**Le réseau de la démo, écrit comme une classe `nn.Module`.**

C'est la façon d'écrire un modèle qu'on garde jusqu'à la fin de la formation : `__init__` déclare les
couches, `forward` dit dans quel ordre les données les traversent.

Le code :
- `nn.Flatten()` : aplatit chaque image en 784 valeurs ;
- `nn.Linear(784, 128)`, `nn.ReLU()`, `nn.Linear(128, 10)` : 128 neurones cachés, puis 10 sorties, un score
  par classe ; ces scores bruts s'appellent des **logits** ;
- `forward` : aplatir, couche cachée, activation, sortie.

In [ ]:
import torch.nn as nn


class ClassifieurMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.aplatir = nn.Flatten()
        self.cachee = nn.Linear(784, 128)
        self.activation = nn.ReLU()
        self.sortie = nn.Linear(128, 10)

    def forward(self, x):
        return self.sortie(self.activation(self.cachee(self.aplatir(x))))

Rien à afficher : la classe est définie, aucun modèle n'est encore créé. `nn.Flatten()` aplatit chaque
image en 784 valeurs, comme `flatten()` dans la démo 2.

**À retenir** : à partir d'ici, chaque modèle de la formation est écrit comme une classe `nn.Module`.

## 9. Modèle, loss et optimiseur

**Le réseau, sa mesure d'erreur et son correcteur.**

C'est la même recette qu'au premier neurone, adaptée à une classification en 10 classes.

Le code :
- `device` : le GPU NVIDIA (`cuda`) dans Colab, le GPU d'un Mac Apple Silicon (`mps`) sur un Mac M1 ou plus récent, sinon le CPU ;
- `ClassifieurMLP()` : crée le modèle, avec des poids tirés au hasard ;
- `.to(device)` : le modèle va sur le GPU ;
- `nn.CrossEntropyLoss()` : la loss de la classification ; elle est faible quand le score de la bonne classe
  domine les autres ;
- `Adam(..., lr=1e-3)` : l'optimiseur de la démo 4.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model = ClassifieurMLP().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Rien à afficher : le modèle, la loss et l'optimiseur sont prêts, sur le GPU.

**Piège fréquent** : créer l'optimiseur avant `model.to(device)`. `model.to` déplace le modèle lui-même,
alors que `tensor.to` renvoie une copie ; l'ordre sûr reste : déplacer le modèle, puis créer l'optimiseur.

## 10. Mesurer la loss et l'accuracy (fourni)

**Deux mesures sur tout un jeu : la loss moyenne et l'accuracy, la part d'images bien classées.**

Avec les deux mesures, on compare l'entraînement et la validation sur la loss comme sur l'accuracy.

Le code :
- `model.eval()` : met le modèle en mode évaluation ;
- `with torch.no_grad():` : on mesure, on n'entraîne pas ;
- `loss_fn(scores, labels).item() * len(labels)` : la loss du lot, multipliée par le nombre d'images, pour une
  moyenne sur tout le jeu ;
- `.argmax(dim=1)` : pour chaque image, la classe au score le plus élevé ; `== labels` compte les bonnes réponses ;
- la fonction renvoie la loss moyenne, puis l'accuracy.

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += loss_fn(scores, labels).item() * len(labels)
            correct += (scores.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Rien à afficher : la fonction est définie. Elle renvoie deux nombres : la loss moyenne, puis l'accuracy.

## 11. Avant l'entraînement

**Quelle accuracy pour un modèle qui n'a encore rien appris ?**

Prédisez : avec 10 classes équilibrées, que donne un modèle qui répond au hasard ?

In [ ]:
loss_val, acc_val = evaluate(model, val_loader, loss_fn)
print(f"acc_val {acc_val:.3f}, loss_val {loss_val:.3f}")

acc_val 0.103 : environ une image sur dix, le hasard avec dix classes. La loss, 2.310, est celle d'un modèle
qui ne sait pas encore départager les classes. Le modèle n'a encore rien appris.

## 12. La boucle : la même qu'au premier neurone

**Les cinq lignes de la démo 1, appliquées à chaque minibatch.**

Rien de nouveau dans la mécanique ; seules les données, le modèle et la loss ont changé. C'est la
force de PyTorch : la même boucle pour un thermomètre et pour des images.

Le code :
- `for epoch in range(1, 6)` : 5 époques, soit 5 passages sur les 50 000 images ;
- `model.train()` : met le modèle en mode entraînement ;
- `for images, labels in train_loader` : un minibatch à la fois ;
- `images.to(device)` : les données vont sur le GPU, comme le modèle ;
- les quatre lignes suivantes : loss, `zero_grad()`, `backward()`, `step()` ;
- `evaluate(model, train_loader, loss_fn)` et `evaluate(model, val_loader, loss_fn)` : loss et accuracy sur
  l'entraînement, puis sur la validation, après chaque époque ;
- `SummaryWriter("runs/fashion")` et `writer.add_scalars(...)` : on écrit les deux loss et les deux accuracies
  de chaque époque, pour TensorBoard.

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/fashion")
for epoch in range(1, 6):
    model.train()
    for images, labels in train_loader:
        loss = loss_fn(model(images.to(device)), labels.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_loader, loss_fn)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
    print(f"époque {epoch}  loss {train_loss:.3f}  acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}")
writer.close()

Une ligne par époque, avec quatre mesures. Dès la première époque, acc_val atteint 0.833 ; à la cinquième,
0.891 sur l'entraînement et 0.874 sur la validation. Les deux loss baissent ensemble, loss_val de 0.456 à 0.341,
et l'écart entre entraînement et validation reste faible : le modèle apprend des motifs qui valent aussi pour des
images qu'il n'a jamais vues.

**À retenir** : le même code que pour le thermomètre apprend à reconnaître des vêtements ; on lit toujours
l'entraînement et la validation ensemble.

**Essayez** : passez la couche cachée à 256 neurones, ou entraînez 10 époques. L'écart entre acc_train et acc_val
se creuse-t-il ?

## 13. Suivre l'entraînement dans TensorBoard

**TensorBoard affiche les courbes écrites pendant l'entraînement.**

Sur un long entraînement, suivre les courbes permet de repérer un problème sans attendre la fin.

Le code :
- `%load_ext tensorboard` : charge l'extension dans le notebook ;
- `%tensorboard --logdir runs` : affiche les courbes du dossier `runs`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

Dans Colab, le tableau de bord de TensorBoard s'affiche sous la cellule, avec la courbe d'accuracy de
validation ; ce notebook enregistré n'en garde pas l'image.

## 14. Voir des prédictions (fourni)

**Huit images de test, avec la classe prédite et la vraie classe.**

Un chiffre global ne dit pas quelles erreurs le modèle fait. Regarder des exemples, c'est vérifier
que le modèle se trompe sur des cas raisonnables.

Le code (à survoler) : on prend le premier lot de test, on prédit les huit premières images, et on les affiche
avec, au-dessus, la classe prédite puis la vraie entre parenthèses.

In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    predictions = model(images[:8].to(device)).argmax(dim=1).cpu()
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(images[i][0], cmap="gray")
    ax.set_title(f"{test_ds.classes[predictions[i]]}\n({test_ds.classes[labels[i]]})", fontsize=8)
    ax.axis("off")
plt.show()

Les huit prédictions sont justes sur ces images : la classe prédite et la vraie classe, entre parenthèses,
coïncident.

**Essayez** : affichez d'autres lots de test ; les erreurs portent-elles sur des classes qui se ressemblent ?

## 15. Une seule mesure sur le jeu de test

**Le modèle est fixé : on le mesure une seule fois sur le jeu de test.**

Si l'on revenait modifier le modèle après avoir vu le test, le test ne serait plus une mesure
honnête de ce que le modèle fera sur de nouvelles images.

In [ ]:
loss_test, acc_test = evaluate(model, test_loader, loss_fn)
print(f"acc_test {acc_test:.3f}, loss_test {loss_test:.3f}")

acc_test 0.868 sur les images de test, jamais vues pendant l'entraînement : proche de l'accuracy de validation.

**À retenir** : les réseaux convolutifs, conçus pour les images et vus dans la suite de la journée, feront mieux.

## À retenir de cette démo

- Un Dataset donne les exemples un par un ; un DataLoader les regroupe en minibatchs mélangés.
- On entraîne sur l'entraînement, on choisit sur la validation, on mesure une seule fois sur le test.
- La boucle est la même que pour le thermomètre ; seuls les données, le modèle et la loss ont changé.